# Regular Expressions — Part 1 · Companion Notebook

**DA2402 · Data Curation and Visualization — Dr. Arun B Ayyar**

This notebook is the runnable version of the **Part 1 — Foundations** deck
(`regex_part1.html`). Every concept taught in the lecture appears here as a cell you
can execute and edit, in the same order as the slides. Section headings carry the
slide numbers, so you can follow along in class or catch up afterwards.

**How to use it**

* Run the cells top to bottom — later sections reuse variables from earlier ones.
* Do not just read the output. Change a pattern, re-run the cell, and see what breaks.



In [20]:
import re
import sys

print("Python", sys.version.split()[0])
print("re module ready")

Python 3.12.13
re module ready


---
## 1.  *slides 2–5*

Real data does not arrive clean. The same phone number shows up written six different
ways, and you are asked to reduce all of them to ten digits.
Compare the with regex and without regex results

In [21]:
def by_hand(s):
    s = s.strip().replace(' ', '').replace('-', '')
    s = s.replace('(', '').replace(')', '')
    if   s.startswith('+91'): s = s[3:]
    elif s.startswith('91') and len(s) == 12: s = s[2:]
    elif s.startswith('0'):   s = s[1:]
    return s if len(s) == 10 and s.isdigit() else None


def with_re(s):
    digits = re.sub(r'\D', '', s)                      # throw away every non-digit
    m = re.fullmatch(r'(?:0|91)?([6-9]\d{9})', digits) # optional prefix, then the number
    return m.group(1) if m else None


tests = ['+91 98765 43210', '09876543210', '98765-43210',
         '(+91)9876543210', '91 9876 543 210', 'Ph: 98765 43210']

print(f"{'input':<22}{'by_hand':<14}{'with_re'}")
print('-' * 50)
for t in tests:
    print(f"{t!r:<22}{str(by_hand(t)):<14}{with_re(t)}")

input                 by_hand       with_re
--------------------------------------------------
'+91 98765 43210'     9876543210    9876543210
'09876543210'         9876543210    9876543210
'98765-43210'         9876543210    9876543210
'(+91)9876543210'     9876543210    9876543210
'91 9876 543 210'     9876543210    9876543210
'Ph: 98765 43210'     None          9876543210


`by_hand` gives up on `'Ph: 98765 43210'` because the stray letters survive its
`replace` chain.

###  *slide 5*

Four ways of using regex

In [22]:
text = "Order 227 shipped 2026-08-11 to PAN ABCDE1234F for Rs 1,299"

print("FIND     ", re.findall(r'\d+', text))
print("VALIDATE ", bool(re.fullmatch(r'\d{6}', '600036')))
print("CLEAN    ", re.sub(r'\s+', ' ', "too    many\n\tspaces"))
print("REDACT   ", re.sub(r'[A-Z]{5}\d{4}[A-Z]', '[REDACTED]', text))

FIND      ['227', '2026', '08', '11', '1234', '1', '299']
VALIDATE  True
CLEAN     too many spaces
REDACT    Order 227 shipped 2026-08-11 to PAN [REDACTED] for Rs 1,299


---
## *slide 6*



In [23]:
m = re.search(r'\d+', "I have 25 apples")

print("match object :", m)
print("m.group()    :", m.group())
print("m.span()     :", m.span())
print("m.start()    :", m.start())
print("m.end()      :", m.end())

match object : <re.Match object; span=(7, 9), match='25'>
m.group()    : 25
m.span()     : (7, 9)
m.start()    : 7
m.end()      : 9


Three rules govern that walk. They explain most regex surprises:

1. **Leftmost wins** — The earliest starting position that can
   match at all.
2. **Then greedy** — having fixed the start, quantifiers take as much as they can while
   still letting the rest of the pattern succeed.
3. **Non-overlapping** — the next search resumes at the previous `.end()`.

In [24]:
s = "aaa aa a"

# leftmost: the engine does not hunt for the longest run, it takes the first one it can start
print("search  ->", re.search(r'a+', s).span(), repr(re.search(r'a+', s).group()))

# non-overlapping: each match resumes where the last one ended
for m in re.finditer(r'a+', s):
    print("  finditer", m.span(), repr(m.group()))

search  -> (0, 3) 'aaa'
  finditer (0, 3) 'aaa'
  finditer (4, 6) 'aa'
  finditer (7, 8) 'a'


---
## 3. *slide 7*

`p` is the **pattern**, `s` the **subject**, `r` the **replacement** (only `sub` takes
one). Below, one `p` and one `s` are run through all seven calls, so the differences
are directly comparable.

In [25]:
p = r'\d{3}-\d{4}'

s = '555-1234 and 555-9876'
r = '[redacted]'

print("re.search   ", re.search(p, s))
print("re.match    ", re.match(p, s))
print("re.fullmatch", re.fullmatch(p, s))
print("re.findall  ", re.findall(p, s))
print("re.finditer ", [m.span() for m in re.finditer(p, s)])
print("re.sub      ", repr(re.sub(p, r, s)))
print("re.compile  ", re.compile(p).findall(s))

re.search    <re.Match object; span=(0, 8), match='555-1234'>
re.match     <re.Match object; span=(0, 8), match='555-1234'>
re.fullmatch None
re.findall   ['555-1234', '555-9876']
re.finditer  [(0, 8), (13, 21)]
re.sub       '[redacted] and [redacted]'
re.compile   ['555-1234', '555-9876']


### The `re.match()` catch

Difference between match and search: `search` and `match` returned the **same
thing**. That is luck — this match happens to start at position 0. Move it off the
start and they part company.

`match` anchors at the beginning but **not** at the end. `fullmatch` is the one that
checks both, which is why it is your default for validation.

In [26]:
p = r'\d{3}-\d{4}'
print("search on 'call 555-1234' ->", re.search(p, 'call 555-1234'))
print("match  on 'call 555-1234' ->", re.match(p, 'call 555-1234'))
print()
print("match     on '555-1234-JUNK' ->", re.match(p, '555-1234-JUNK'))
print("fullmatch on '555-1234-JUNK' ->", re.fullmatch(p, '555-1234-JUNK'))

search on 'call 555-1234' -> <re.Match object; span=(5, 13), match='555-1234'>
match  on 'call 555-1234' -> None

match     on '555-1234-JUNK' -> <re.Match object; span=(0, 8), match='555-1234'>
fullmatch on '555-1234-JUNK' -> None


### The `re.findall()`

If your pattern contains capturing groups, `findall` hands back the **groups**, not the
whole match. One group gives a list of strings; two or more gives a list of tuples.
This bites everyone exactly once.

In [27]:
print("no group   ", re.findall(r'\d{3}-\d{4}',     s))
print("one group  ", re.findall(r'(\d{3})-\d{4}',   s))
print("two groups ", re.findall(r'(\d{3})-(\d{4})', s))

no group    ['555-1234', '555-9876']
one group   ['555', '555']
two groups  [('555', '1234'), ('555', '9876')]


---
## 4. Always write patterns as `r'…'` — *slide 8*

`\b` means "word boundary" to the regex engine, but `\b` means **backspace** to Python's
string parser. Without the `r` prefix, Python eats the escape before `re` ever sees it —
and you get no error, just silence.

In [28]:
plain = '\bcat\b'
raw   = r'\bcat\b'

print("what Python stores for '\\bcat\\b' :", repr(plain))
print("what Python stores for r'\\bcat\\b':", repr(raw))
print()
print("re.findall(plain, 'the cat sat') ->", re.findall(plain, 'the cat sat'))
print("re.findall(raw,   'the cat sat') ->", re.findall(raw,   'the cat sat'))

what Python stores for '\bcat\b' : '\x08cat\x08'
what Python stores for r'\bcat\b': '\\bcat\\b'

re.findall(plain, 'the cat sat') -> []
re.findall(raw,   'the cat sat') -> ['cat']


Neither line raised an exception. The broken one simply found nothing. **Write every
pattern as a raw string, every time** — it costs one character and removes a whole class
of silent bug.

---
## 5. The syntax — *slides 9–12*

### 5.1 Metacharacters — *slide 9*

These characters do not stand for themselves. To match one literally, escape it with a
backslash.

In [29]:
demo = "a.b a1b aXb ab abb"

print(". any char except newline ", re.findall(r'a.b', demo))
print("\\. a literal dot          ", re.findall(r'a\.b', demo))
print("^ start of string         ", re.findall(r'^a.b', demo))
print("$ end of string           ", re.findall(r'b$', demo))
print("| alternation             ", re.findall(r'cat|dog', 'a dog and a cat'))
print("() group                  ", re.search(r'(ab)+', 'ababab').group(0))

. any char except newline  ['a.b', 'a1b', 'aXb', 'abb']
\. a literal dot           ['a.b']
^ start of string          ['a.b']
$ end of string            ['b']
| alternation              ['dog', 'cat']
() group                   ababab


### 5.2 Character classes — one class matches **one** character — *slide 10*

A class is a menu for a single position. `[aeiou]` is not "a vowel sequence", it is
"one character, and it must be one of these five".

In [30]:
print("[aeiou]    ", re.findall(r'[aeiou]',   'curation'))
print("[^aeiou]   ", re.findall(r'[^aeiou]',  'curation'))   # ^ inside [] means NOT
print("[A-Za-z]+  ", re.findall(r'[A-Za-z]+', 'DA2402 rocks'))
print()
print(r"\d digit    ", re.findall(r'\d', 'a1b22'))
print(r"\w word     ", re.findall(r'\w', 'a-1_!'))
print(r"\s space    ", re.findall(r'\s', 'a b\tc'))
print(r"\D \W \S are the negations:", re.findall(r'\D', 'a1b22'))

[aeiou]     ['u', 'a', 'i', 'o']
[^aeiou]    ['c', 'r', 't', 'n']
[A-Za-z]+   ['DA', 'rocks']

\d digit     ['1', '2', '2']
\w word      ['a', '1', '_']
\s space     [' ', '\t']
\D \W \S are the negations: ['a', 'b']


### 5.3 Quantifiers — *slide 11*

A quantifier applies to the thing immediately before it. By default quantifiers are
**greedy**: they take as much as they can and give back only if forced. Add `?` to make
one **lazy**, so it takes as little as possible.



In [31]:
html = '<b>one</b><b>two</b>'

print("greedy <.*>  ->", re.findall(r'<.*>',  html))   # one match, the whole line
print("lazy   <.*?> ->", re.findall(r'<.*?>', html))   # each tag separately
print()
print("*  zero or more ", re.findall(r'ab*', 'a ab abb'))
print("+  one or more  ", re.findall(r'ab+', 'a ab abb'))
print("?  zero or one  ", re.findall(r'ab?', 'a ab abb'))
print("{2,3} a range   ", re.findall(r'a{2,3}', 'a aa aaa aaaa'))

greedy <.*>  -> ['<b>one</b><b>two</b>']
lazy   <.*?> -> ['<b>', '</b>', '<b>', '</b>']

*  zero or more  ['a', 'ab', 'abb']
+  one or more   ['ab', 'abb']
?  zero or one   ['a', 'ab', 'ab']
{2,3} a range    ['aa', 'aaa', 'aaa']


### 5.4 Anchors match positions, not characters — *slide 12*

Anchors are **zero-width**: they consume nothing. `^`, `$`, `\b` and `\B` assert
something about the *gap between* characters.

A word boundary `\b` sits wherever a `\w` is adjacent to a `\W` or a string edge.

In [32]:
subject = 'the cat scattered'

print("\\bcat\\b ->", re.findall(r'\bcat\b', subject))   # only the standalone word
print("plain cat ->", re.findall(r'cat',      subject))   # also the one inside 'scattered'
print()
for m in re.finditer(r'\bcat\b', subject):
    print("  matched", repr(m.group()), "at", m.span())

# ^ and $ under MULTILINE
lines = "alpha\nbeta\ngamma"
print()
print("^\\w+ plain     ", re.findall(r'^\w+', lines))
print("^\\w+ MULTILINE ", re.findall(r'^\w+', lines, re.MULTILINE))

\bcat\b -> ['cat']
plain cat -> ['cat', 'cat']

  matched 'cat' at (4, 7)

^\w+ plain      ['alpha']
^\w+ MULTILINE  ['alpha', 'beta', 'gamma']


**Run this.** `re.findall(r'\Bcat\B', 'the cat scattered')` — what comes back, and why?
Predict before running the next cell.

In [33]:
# \B asserts NOT a boundary, so it only matches 'cat' when it is buried inside a word
print(re.findall(r'\Bcat\B', 'the cat scattered'))

['cat']


---
## 6.  — *slides 14–16*



In [34]:
def find_digits(text):
    return re.findall(r'\d+', text)

def find_words(text):
    return re.findall(r'[a-zA-Z]+', text)

def find_emails(text):
    return re.findall(r'[\w.-]+@[\w.-]+\.\w+', text)

def validate_phone(phone):
    pattern = r'^\(?\d{3}\)?[-\s.]?\d{3}[-\s.]?\d{4}$'
    return bool(re.match(pattern, phone))


text1  = "I have 25 apples and 30 oranges. Call me at 555-1234."
text2  = "Contact john.doe@email.com or jane_smith@company.org."
phone1 = "(555) 123-4567"
phone2 = "555.123.4567"
phone3 = "123-45-6789"        # wrong shape

print("digits :", find_digits(text1))
print("words  :", find_words(text1))
print("emails :", find_emails(text2))
print("phones :", validate_phone(phone1), validate_phone(phone2), validate_phone(phone3))

digits : ['25', '30', '555', '1234']
words  : ['I', 'have', 'apples', 'and', 'oranges', 'Call', 'me', 'at']
emails : ['john.doe@email.com', 'jane_smith@company.org']
phones : True True False


### Read a pattern left to right — *slide 16*

Take the phone validator apart. Every pattern you meet can be read this way.

| piece | job |
|---|---|
| `^` | anchor at the start of the string |
| `\(?` | an optional literal `(` — escaped, because `(` normally groups |
| `\d{3}` | exactly three digits |
| `\)?` | an optional literal `)` |
| `[-\s.]?` | one optional separator: hyphen, whitespace or dot |
| `\d{3}` | three more digits |
| `[-\s.]?` | another optional separator |
| `\d{4}` | exactly four digits |
| `$` | anchor at the end |

Note what it *assumes*: exactly 3-3-4 digits, and at most one separator between blocks.
It accepts `(555 123-4567` — an unbalanced bracket — because `\(?` and `\)?` are
independent. Patterns are only as strict as you make them.

In [35]:
print("unbalanced bracket accepted?", validate_phone("(555 123-4567"))

unbalanced bracket accepted? True


---
## 7. Exercise 2 — from finding to changing — *slides 17–19*

`re.sub` is the other half of the module. Same patterns, but now you rewrite instead of
extract.

In [36]:
def clean_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

def extract_urls(text):
    return re.findall(r'https?://[\w.-]+(?:/[\w.-]*)*', text)

def remove_html_tags(text):
    return re.sub(r'<.*?>', '', text)        # lazy: one tag at a time

def extract_hashtags(text):
    return re.findall(r'#\w+', text)


messy_text  = "Hello    world!\n\n\tThis   has    weird\tspacing."
url_text    = "Visit https://example.com or http://test.org for more info."
html_text   = "<p>This is <strong>bold</strong> and <em>italic</em> text.</p>"
social_text = "Loving this #python tutorial! #coding #learning #webdev"

print("whitespace:", repr(clean_whitespace(messy_text)))
print("urls      :", extract_urls(url_text))
print("html      :", repr(remove_html_tags(html_text)))
print("hashtags  :", extract_hashtags(social_text))

whitespace: 'Hello world! This has weird spacing.'
urls      : ['https://example.com', 'http://test.org']
html      : 'This is bold and italic text.'
hashtags  : ['#python', '#coding', '#learning', '#webdev']


Note `remove_html_tags` uses the **lazy** `<.*?>`. With the greedy `<.*>` it would
swallow everything from the first `<` to the last `>` — the whole line.

In [37]:
print("lazy  :", repr(re.sub(r'<.*?>', '', html_text)))
print("greedy:", repr(re.sub(r'<.*>',  '', html_text)))

lazy  : 'This is bold and italic text.'
greedy: ''


### `re.sub` accepts a function — *slide 19*

When the replacement depends on what was matched, pass a callable. It receives the Match
object and returns the replacement string. This is how you reformat rather than just
delete.

In [38]:
def normalize_phone_numbers(text):
    pattern = r'\(?([0-9]{3})\)?[-\s.]?([0-9]{3})[-\s.]?([0-9]{4})'

    def replace_phone(match):
        return f'({match.group(1)}) {match.group(2)}-{match.group(3)}'

    return re.sub(pattern, replace_phone, text)


phone_text = "Call 555-123-4567 or (555) 987-6543 or 555.111.2222"
print(normalize_phone_numbers(phone_text))

Call (555) 123-4567 or (555) 987-6543 or (555) 111-2222


For simple rearrangements you do not even need a function — `\1`, `\2` in the
replacement string refer back to the capturing groups.

In [39]:
print(re.sub(r'(\w+), (\w+)', r'\2 \1', 'Biology, Class'))

Class Biology


---
## 8. Groups — parentheses do two separate jobs — *slides 20–21*

Parentheses **bind** (so a quantifier can apply to more than one character) and they
**capture** (so you can get a piece back). You often want one without the other.

| form | binds | captures | retrieve with |
|---|---|---|---|
| `(…)` | yes | yes, numbered | `.group(1)` |
| `(?:…)` | yes | **no** | — |
| `(?P<name>…)` | yes | yes, named | `.group('name')` |

In [40]:
m = re.search(r'(?P<y>\d{4})-(?P<m>\d{2})-(?P<d>\d{2})', 'due 2026-08-11')

print("group('y')  :", m.group('y'))
print("groups()    :", m.groups())
print("groupdict() :", m.groupdict())

group('y')  : 2026
groups()    : ('2026', '08', '11')
groupdict() : {'y': '2026', 'm': '08', 'd': '11'}


Name your groups once a pattern has more than two.

### 8.1 Capturing vs non-capturing — one example, both ways — *slide 21*

`(\.\d{2})?` becomes `(?:\.\d{2})?`. **The text that matches is identical. Only what you can retrieve
changes.**

In [41]:
s2  = 'items: $45.49, $7, $1200.00'
cap = r'\$(\d+)(\.\d{2})?'      # capturing
non = r'\$(\d+)(?:\.\d{2})?'    # non-capturing

mc, mn = re.search(cap, s2), re.search(non, s2)

print("SAME MATCH")
print("  capturing     group(0), span:", repr(mc.group(0)), mc.span())
print("  non-capturing group(0), span:", repr(mn.group(0)), mn.span())
print()
print("DIFFERENT BOOKKEEPING")
print("  capturing     .groups() :", mc.groups())
print("  non-capturing .groups() :", mn.groups())
print("  capturing     findall   :", re.findall(cap, s2))
print("  non-capturing findall   :", re.findall(non, s2))
print("  group count             :", re.compile(cap).groups, "vs", re.compile(non).groups)

SAME MATCH
  capturing     group(0), span: '$45.49' (7, 13)
  non-capturing group(0), span: '$45.49' (7, 13)

DIFFERENT BOOKKEEPING
  capturing     .groups() : ('45', '.49')
  non-capturing .groups() : ('45',)
  capturing     findall   : [('45', '.49'), ('7', ''), ('1200', '.00')]
  non-capturing findall   : ['45', '7', '1200']
  group count             : 2 vs 1


In [42]:
# asking for a group that does not exist
try:
    mn.group(2)
except IndexError as e:
    print("non-capturing .group(2) ->", type(e).__name__ + ":", e)

# and the same in a replacement string
try:
    re.sub(non, r'INR \2', s2)
except re.error as e:
    print("non-capturing sub \\2   -> re.error:", e)

non-capturing .group(2) -> IndexError: no such group
non-capturing sub \2   -> re.error: invalid group reference 2 at position 5


Two details worth remembering, both visible above:

* An optional group that never matched is reported as `''` by `findall` but as `None`
  by `.groups()` — the same absence, two answers.
* A capturing group **inside a quantifier keeps only its last repetition**.

In [43]:
print("findall (?:ab)+ :", re.findall(r'(?:ab)+', 'ababab xx abab'))
print("findall (ab)+   :", re.findall(r'(ab)+',   'ababab xx abab'))
print()
for p_ in (r'(?:ab)+', r'(ab)+'):
    print(f"  {p_:<10}", [(m.span(), m.group(0)) for m in re.finditer(p_, 'ababab xx abab')])

findall (?:ab)+ : ['ababab', 'abab']
findall (ab)+   : ['ab', 'ab']

  (?:ab)+    [((0, 6), 'ababab'), ((10, 14), 'abab')]
  (ab)+      [((0, 6), 'ababab'), ((10, 14), 'abab')]


Both match the identical runs at `(0, 6)` and `(10, 14)` — but `(ab)+` records only the
final `ab`, and `findall` reports the capture rather than the match.

**Rule of thumb:** use `(?:…)` whenever you need the parentheses only to bind, and keep
`(…)` for the pieces you actually intend to read back.

---
## 9. Exercise 3 —  *slides 22–24*

Structured extraction: one pattern that describes the whole line, with groups around the
fields you want.

### 9.1 An Apache log line, six fields

In [44]:
def parse_log_entry(log):
    pattern = (r'([\d.]+) - - \[([^\]]+)\] '
               r'"(\w+) ([^\s]+) [^"]+" (\d+) (\d+)')
    match = re.search(pattern, log)
    if match:
        return {'ip':     match.group(1),
                'date':   match.group(2),
                'method': match.group(3),
                'path':   match.group(4),
                'status': match.group(5),
                'size':   match.group(6)}
    return None


log_entry = ('192.168.1.1 - - [10/Oct/2023:13:55:36 +0000] '
             '"GET /index.html HTTP/1.1" 200 2326')

for k, v in parse_log_entry(log_entry).items():
    print(f"  {k:<7}: {v}")

  ip     : 192.168.1.1
  date   : 10/Oct/2023:13:55:36 +0000
  method : GET
  path   : /index.html
  status : 200
  size   : 2326


### 9.2 Three date formats, one normalised output

In [45]:
def extract_dates(text):
    dates = []
    months = {'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04',
              'May': '05', 'Jun': '06', 'Jul': '07', 'Aug': '08',
              'Sep': '09', 'Oct': '10', 'Nov': '11', 'Dec': '12'}

    for m in re.finditer(r'(\d{4})-(\d{1,2})-(\d{1,2})', text):          # 2023-10-15
        y, mo, d = m.groups()
        dates.append(f"{y}-{mo.zfill(2)}-{d.zfill(2)}")

    for m in re.finditer(r'(\d{1,2})/(\d{1,2})/(\d{4})', text):          # 10/20/2023
        mo, d, y = m.groups()
        dates.append(f"{y}-{mo.zfill(2)}-{d.zfill(2)}")

    for m in re.finditer(r'([A-Za-z]{3})\w*\s+(\d{1,2}),\s+(\d{4})', text):  # Oct 25, 2023
        mon, d, y = m.groups()
        dates.append(f"{y}-{months[mon.title()]}-{d.zfill(2)}")

    return sorted(dates)


date_text = "Meeting on 2023-10-15, deadline is 10/20/2023, and party on Oct 25, 2023"
print(extract_dates(date_text))

['2023-10-15', '2023-10-20', '2023-10-25']


### 9.3 A CSV line where the fields contain commas

Alternation does the work: *a quoted field* **or** *an unquoted one*. Whichever branch
matched is the one that is not `None`.

In [46]:
def parse_csv_line(line):
    fields = []
    for m in re.finditer(r'"([^"]*)"|([^,]+)', line):
        field = m.group(1) if m.group(1) is not None else m.group(2)
        fields.append(field.strip())
    return fields


csv_line = 'John,"Doe, Jr.",30,"New York, NY",Engineer'
print(parse_csv_line(csv_line))

['John', 'Doe, Jr.', '30', 'New York, NY', 'Engineer']


### 9.4 Password strength — independent rules, scored

Do **not** try to write one pattern for this. Five small independent checks are readable,
testable and give a useful error message.

In [47]:
def check_password(password):
    requirements = {
        'length':    len(password) >= 8,
        'lowercase': bool(re.search(r'[a-z]', password)),
        'uppercase': bool(re.search(r'[A-Z]', password)),
        'digit':     bool(re.search(r'\d', password)),
        'special':   bool(re.search(r'[!@#$%^&*(),.?":{}|<>]', password)),
    }
    return {'valid':   all(requirements.values()),
            'score':   sum(requirements.values()),
            'missing': [k for k, ok in requirements.items() if not ok]}


for pw in ["MyP@ssw0rd123", "password"]:
    print(f"{pw!r:<18}", check_password(pw))

'MyP@ssw0rd123'    {'valid': True, 'score': 5, 'missing': []}
'password'         {'valid': False, 'score': 2, 'missing': ['uppercase', 'digit', 'special']}


---
## 10. Run it. The comment is not evidence. — *slide 25*

Preparing the lecture meant executing every cell of the tutorial notebook, and several
comments turned out to disagree with what Python actually returns. Here is one, live:
a pattern whose comment claims it strips articles.

In [48]:
claim = "matches words not preceded by 'a', 'an' or 'the'"
try:
    re.findall(r'(?<!\b(?:a|an|the)\s)\b[a-zA-Z]+', "the cat and a dog", re.IGNORECASE)
except re.error as e:
    print("the comment said:", claim)
    print("what Python said: re.error:", e)

the comment said: matches words not preceded by 'a', 'an' or 'the'
what Python said: re.error: look-behind requires fixed-width pattern


The comment was written by someone who never ran the cell. **Run the cell.**

In [49]:
try:
    re.compile(r'(?<!\b(?:a|an|the)\s)\b[a-zA-Z]+')
except re.error as e:
    print("re.error:", e)

# lookAHEAD has no such restriction — variable width is fine there
print("variable-width lookahead compiles fine:", bool(re.compile(r'\w+(?=\s(?:cat|kitten))')))

re.error: look-behind requires fixed-width pattern
variable-width lookahead compiles fine: True


In [50]:
def words_not_after_articles(text):
    tokens = re.findall(r'[a-zA-Z]+', text)
    articles = {'a', 'an', 'the'}
    return [w for i, w in enumerate(tokens)
            if i == 0 or tokens[i - 1].lower() not in articles]


sentence = "the cat and a dog met an owl"
kept = words_not_after_articles(sentence)
print("all words :", re.findall(r'[a-zA-Z]+', sentence))
print("kept      :", kept)
print("dropped   :", [w for w in re.findall(r'[a-zA-Z]+', sentence) if w not in kept])

all words : ['the', 'cat', 'and', 'a', 'dog', 'met', 'an', 'owl']
kept      : ['the', 'and', 'a', 'met', 'an']
dropped   : ['cat', 'dog', 'owl']


---
## 12. Exercise 5 — messy records in, validated records out — *slide 28*

Everything so far, on one problem. Note the shape of the solution: **many small named
patterns**, applied independently, rather than one enormous pattern.

In [51]:
PATTERNS = {
    'name':   r'(?:Customer|CUSTOMER|Name|Mr\.|Ms\.|Mrs\.)[:\s]+([A-Za-z][A-Za-z.\s]+?)(?=,)',
    'phone':  r'(?:Phone|PHONE|Tel)[:\s]*(\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4})',
    'email':  r'(?:Email|EMAIL)[:\s]*([\w.+-]+@[\w-]+\.[A-Za-z]{2,})',
    'amount': r'\$\s?(\d+(?:\.\d{2})?)',
    'iso':    r'(\d{4}-\d{2}-\d{2})',
}

def parse_record(rec):
    out = {}
    for field, pat in PATTERNS.items():
        m = re.search(pat, rec)
        out[field] = m.group(1).strip() if m else None
    out['phone_valid'] = out['phone'] is not None
    out['email_valid'] = out['email'] is not None
    out['valid_record'] = all([out['name'], out['phone_valid'],
                               out['email_valid'], out['amount']])
    return out


records = [
    "Customer: John Smith, Phone: (555) 123-4567, Email: john@email.com, "
    "Address: 123 Main St, New York, NY 10001, Purchase: $299.99 on 2023-10-15",

    "CUSTOMER: Bob Wilson, PHONE: 555-111-2222, EMAIL: bob@invalid, "
    "Address: 789 Pine Rd, Purchase: $75.50 on 2023-10-25",
]

for i, rec in enumerate(records, 1):
    print(f"--- record {i} ---")
    for k, v in parse_record(rec).items():
        print(f"  {k:<13}: {v}")

--- record 1 ---
  name         : John Smith
  phone        : (555) 123-4567
  email        : john@email.com
  amount       : 299.99
  iso          : 2023-10-15
  phone_valid  : True
  email_valid  : True
  valid_record : True
--- record 2 ---
  name         : Bob Wilson
  phone        : 555-111-2222
  email        : None
  amount       : 75.50
  iso          : 2023-10-25
  phone_valid  : True
  email_valid  : False
  valid_record : False


Record 2's email is `bob@invalid` — no dot, no top-level domain — so the email pattern
does not match and the record is correctly flagged invalid. The pattern did not have to
"know" what a bad email is; it only had to describe a good one.

---
## 13. Where not to use regex — *slide 29*

1. **Nested or recursive structure** — HTML, JSON, source code, balanced brackets. Regex
   has no memory of depth and provably cannot count them.
2. **A real parser already exists** — use `csv`, `json`, `email.utils`, `urllib.parse`,
   `datetime.strptime`. They handle the edge cases you have not thought of yet.
3. **The pattern is longer than the code that would replace it** — if nobody on your team
   can read it in six months, it is a liability.
4. **You are validating something with a specification** — email, URL, IBAN. Use a
   library that tracks the spec.

Here is failure mode 1, live.

In [52]:
nested = "<div><p>hello</p></div>"

# a regex cannot tell which </div> closes which <div>
print("regex 'inner text':", re.findall(r'<div>(.*)</div>', nested))
print("regex tag stripper:", re.sub(r'<.*?>', '', nested))

# the moment the structure nests unevenly, any regex answer becomes guesswork
broken = "<div><p>a</div></p>"
print("still 'works', still wrong:", re.sub(r'<.*?>', '', broken))

regex 'inner text': ['<p>hello</p>']
regex tag stripper: hello
still 'works', still wrong: a


Both produce output. Neither *understands* the document

---
## 14. What to practise before Part 2 — *slide 30*

* Rewrite one of your own text-cleaning scripts using `re.sub` with a function.
* Take any pattern above, break it deliberately, and predict the output before running.



### Quick reference — everything used in this notebook

| pattern | meaning |
|---|---|
| `\d \w \s` | digit, word char, whitespace (`\D \W \S` negate) |
| `[abc]` `[^abc]` | one of these / anything but these |
| `*` `+` `?` `{n,m}` | quantifiers — greedy by default, add `?` for lazy |
| `^` `$` `\b` `\B` | anchors — zero width |
| `(…)` `(?:…)` `(?P<n>…)` | capture / bind only / named capture |
| `(?=…)` `(?!…)` `(?<=…)` `(?<!…)` | lookaround — zero width |
| `re.search / match / fullmatch` | first match / anchored at 0 / the whole string |
| `re.findall / finditer` | all matches as values / as Match objects |
| `re.sub(p, r, s)` | replace; `r` may be a string with `\1` or a function |
| `re.compile(p)` | reuse a pattern, and put flags in one place |